# Hierarchical Clustering - UPGMA

---
## Learning Objectives

1. Implement a distance-based phylogeny neighbor joining algorithm
1. Understanding of hierarchical clustering
1. Manipulation of arrays and lists in Python


---
## Imports
You will need to install a two new packages to properly render your phylogenetic trees. Using your preferred package manager, install both [`biopython`](https://biopython.org/) and `matplotlib` into your virtual environments.

For example:
```bash
$ micromamba create -n module06 -c bioconda -c conda-forge python numpy matplotlib biopython
```

In [ ]:
from typing import List, Tuple, Dict
from io import StringIO

import matplotlib.pyplot as plt
from Bio import Phylo
import numpy as np

---
## Background

Today we will implement a distance-based phylogenetic tree construction method using the Neighbor-Joining (NJ) algorithm and Smith-Waterman local alignment scores. Unlike UPGMA (Unweighted Pair Group Method using Arithmetic averages), NJ produces unrooted trees and does not assume a constant evolutionary rate across lineages, making it more biologically realistic for analyzing sequence relationships.

We will use HIV-1 reverse transcriptase sequences to construct our phylogenetic tree. The Smith-Waterman algorithm will be used to generate pairwise local alignment scores, which will then be converted into distances for tree construction. This approach is particularly suitable for HIV sequence analysis as it:
1. Handles sequence variations effectively through local alignment
2. Accounts for potential rate heterogeneity across different viral strains
3. Does not assume a molecular clock

The ultimate output will be an unrooted phylogenetic tree representing the evolutionary relationships between the HIV-1 sequences, visualized using the ete3 library. This method provides insights into viral diversity and evolutionary patterns while avoiding the assumptions of simpler hierarchical clustering approaches.

The key innovations of this implementation are:
- Use of Smith-Waterman for sensitive local alignment scoring
- Implementation of Neighbor-Joining for unrooted tree construction
- More biologically realistic evolutionary model

## Neighbor-Joining Algorithm

**Input**: Distance matrix $D$ for $n$ sequences

**Initialization**:
- Let $n$ be the number of sequences
- Assign each sequence $i$ to its own leaf node
- Initialize tree $T$ with leaf nodes

**Iteration**:
While $n > 2$:
1. Calculate Q-matrix where:
   $Q(i,j) = (n-2)d(i,j) - \sum_{k=1}^n d(i,k) - \sum_{k=1}^n d(j,k)$

2. Find pair $(i,j)$ with minimum $Q(i,j)$

3. Calculate branch lengths:
   $d_i = \frac{d(i,j) + (r_i - r_j)/(n-2)}{2}$
   $d_j = d(i,j) - d_i$
   where $r_i = \sum_{k=1}^n d(i,k)$

4. Create new node $k$
   - Add branches from $k$ to $i$ and $j$ with lengths $d_i$ and $d_j$

5. Update distances to remaining nodes $x$:
   $d(k,x) = \frac{d(i,x) + d(j,x) - d(i,j)}{2}$

6. Remove nodes $i$ and $j$
7. Add node $k$ to active nodes
8. $n = n - 1$

**Termination**:
When $n = 2$ with remaining nodes $i$ and $j$:
- Add final branch between $i$ and $j$ with length $d(i,j)$
- Return unrooted tree $T$

---
## Distance metrics for comparing sequences

### Smith-Waterman
This was previously implemented. Feel free to import if you know how...or Copy & Paste.



In [ ]:
def read_fasta(filename: str) -> Dict[str, str]:
    """Reads sequences from FASTA file.

    Args:
        filename (str): Path to FASTA file containing HIV RT sequences

    Returns:
        Dict[str, str]: Dictionary mapping sequence IDs to sequences

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> len(seqs) > 0
        True
    """
    # initialize
    seqs = {}
    curr_name = ""
    curr_seq = ""

    # open the file
    with open(filename, mode="r", encoding="utf-8") as infile:
        # iterate over lines
        for line in infile:
            # check if line is header
            if ">" in line:

                # this will run on the first line, but also any time we start a new sequence
                # so if this isn't the first line in the file, we'll have a sequence to pack into the dict here
                if curr_name and curr_seq:
                    # this tells us the last sequence we were looking at is now done, so we can pack it up
                    seqs[curr_name] = curr_seq

                    # now clear the current sequence
                    curr_seq = ""

                # and update the title for the next sequence we're about to look at
                curr_name = line.strip()

            else:
                # here, the current line isn't a header, so it's just a part of a sequence
                # add the current line to the current sequence we're building
                curr_seq += line.strip()
    
    # report the dict of sequences
    return seqs


In [ ]:
from textdistance import hamming, smith_waterman

"""
From our insightful meeting with Marcus, we learned about the textdistance package. 

Textdistance has ~30 in-built algorithms for calculating sequence similarity and distance, including one for Smith Waterman
The documentation for this package is a little lacking, but some toying around in the python console revealed that
the distance it calculates for two sequences is essentially just the similarity score for the two sequences subtracted 
from the sequence length. The normalized versions of the similarity and distance scores are then generated by dividing
the similarity or distance by the length of the aligned sequences. 


TODO: add explanation about how it makes a Smith-Waterman alignment matrix to calc the scores and the matrix itself is where you para-
meterize stuff like mismatch and gap penalties
"""

In [ ]:
def build_distance_matrix(sequences: Dict[str, str]) -> Tuple[np.ndarray, List[str]]:
    """Builds DISTANCE (not similiarity) matrix using Smith-Waterman scores.

    Args:
        sequences (Dict[str, str]): Dictionary of HIV RT sequences

    Returns:
        Tuple[np.ndarray, List[str]]: Distance matrix and list of sequence IDs

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> mat[0:3, 0:3]  # Show 3x3 slice of distance matrix
        array([[0.000, 0.004, 0.012],
              [0.004, 0.000, 0.012],
              [0.012, 0.012, 0.000]])
        >>> len(ids)  # Number of sequences
        19
    """
    # initialize an N x N matrix, where N is number of sequences
    # use the same dtype as the output from the textdistance.smith_waterman function
    dist_matrix = np.zeros(shape=(len(sequences), len(sequences)), dtype=np.float64)

    # initialize variables tracking the current row and column we're on in the distance matrix
    seq_ids = []
    # iterate over indices and the sequence IDs of the sequences dict
    for nrow, row_id in enumerate(sequences.keys()):
        for ncol, col_id in enumerate(sequences.keys()):
            
            # check if we've already hit this combination (scores are symmetrical and initialize at 0)
            # we hit position (nrow, ncol) before we hit position (ncol, nrow)
            # also check if we're comparing a sequence to itself- since the distance is always 0, we can skip that calculation
            if dist_matrix[nrow][ncol] == 0 and nrow != ncol:

                # get the distance score for the current combination of sequences (between 0 and 1)
                curr_dist = smith_waterman.normalized_distance(sequences[row_id], sequences[col_id])

                # put the current score into the associated positions on the graph
                # the dist between 2 sequences will be the same regardless of which one you start measuring from
                dist_matrix[nrow][ncol] = curr_dist
                dist_matrix[ncol][nrow] = curr_dist

        # add the current species name to the list of seqIDs
        seq_ids.append(row_id)
    
    # report the distance matrix
    return dist_matrix, seq_ids

In [ ]:
from tree_object_utils import Node, Tree_Graph

def neighbor_joining(
    distance_matrix: np.ndarray,
    labels: List[str]
) -> Tree_Graph:
    """Implements Neighbor-Joining algorithm for phylogenetic tree construction.

    Args:
        distance_matrix (np.ndarray): Distance matrix from Smith-Waterman scores
        labels (List[str]): Sequence identifiers

    Returns:
        str: Newick format tree string

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> tree = neighbor_joining(mat, ids)
        >>> tree.startswith("((DM1:0.002,DM2:0.002)")  # Start of Newick string
        True
        >>> tree.count(",")  # Number of separators in tree
        18
    """

  #copying the distance matrix
    D = distance_matrix.copy().astype(float)
    labels = list(labels)
    n = len(labels)

    # initialize the graph
    graph = Tree_Graph(D)
    graph.matrix_labels = labels

    # base case
    # if and when 2 sequences are left, make leaf nodes and stop recursing
    if n == 2:
        branch_length = D[0, 1] / 2.0

        node1 = Node(labels[0], branch_length)
        node2 = Node(labels[1], branch_length)

        graph.nodes[node1.seq_id] = node1
        graph.nodes[node2.seq_id] = node2

        if node1 not in graph.leaves:
            graph.leaves.append(node1)
        if node2 not in graph.leaves:
            graph.leaves.append(node2)

        # storing these so the previous recursive call can utilise them
        graph.last_nodes = [node1, node2]

        return graph

    # node selection
    max_dist = -np.inf
    candidate_a = 0
    candidate_b = 1

    for i in range(n):
        for j in range(i+1, n):
            if D[i, j] > max_dist:
                max_dist = D[i, j]
                candidate_a = i
                candidate_b = j

    # pick the third candidate from the rest of the sequences
    rest = [k for k in range(n) if k not in (candidate_a, candidate_b)]
    candidate_c = rest[0]

    # trim the sequence that is less related to the third sequence
    if D[candidate_a, candidate_c] >= D[candidate_b, candidate_c]:
        trim = candidate_a
        other = candidate_b
    else:
        trim = candidate_b
        other = candidate_a

    # calculate branch length
    row_sums = D.sum(axis=1)
    branch_length = (
        D[trim, other] / 2.0
        + (row_sums[trim] - row_sums[other]) / (2.0 * (n - 2))
    )
    if branch_length < 0:
        branch_length = 0.0

    trim_node = Node(labels[trim], branch_length)
    trim_node.branch_length = branch_length

    # subtract branch length from all distances for the trimmed sequence
    for k in range(n):
        if k != trim:
            trim_node.balded_distances[labels[k]] = D[trim, k] - branch_length

    # trim the matrix
    retain         = [k for k in range(n) if k != trim]
    D_trimmed      = D[np.ix_(retain, retain)]
    labels_trimmed = [labels[k] for k in retain]

    # recursion
    graph = neighbor_joining(D_trimmed, labels_trimmed)

    # UNWINDING!
    child1, child2 = graph.last_nodes

    # add trimmed leaf to graph
    graph.nodes[trim_node.seq_id] = trim_node
    if trim_node not in graph.leaves:
        graph.leaves.append(trim_node)

    # compute branch length to find how distant a node's parent is from every other leaf
    # how far the parent is from child1
    d_px = trim_node.balded_distances.get(child1.seq_id, 0.0)
    # how far the parent is from child2
    d_py = trim_node.balded_distances.get(child2.seq_id, 0.0)

    #check how far child1 and child2 are in the distance matrix
    if child1.seq_id in graph.matrix_labels and child2.seq_id in graph.matrix_labels:
        ix   = graph.matrix_labels.index(child1.seq_id)
        iy   = graph.matrix_labels.index(child2.seq_id)
        d_xy = graph.distance_matrix[ix][iy]
    else:
        d_xy = 0.0

    #calculating the parent branch length
    parent_branch = max((d_px + d_py - d_xy) / 2.0, 0.0)

    # build parent name as Newick-style label
    parent_name = ",".join(str(child) for child in [trim_node, child1, child2])

    # create the internal parent node
    parent_node = Node(parent_name, parent_branch)
    graph.nodes[parent_node.seq_id] = parent_node

    # connect parent to its children
    graph.add_edge(parent_node, trim_node)
    graph.add_edge(parent_node, child1)
    graph.add_edge(parent_node, child2)

    # update last_nodes so the upper recursive call can continue
    graph.last_nodes = [trim_node, parent_node]

    return graph

In [ ]:
def plot_tree(newick_tree: str) -> None:
    """Plots a phylogenetic tree from a Newick string using Biopython and Matplotlib.

    The tree is rendered in a rectangular layout with smaller leaf labels and
    added margins to reduce overlap between labels and branches.

    Args:
        newick_tree (str): Tree in Newick format

    Returns:
        None: Displays the plotted tree
    """
    handle = StringIO(newick_tree)
    tree = Phylo.read(handle, "newick")

    fig, ax = plt.subplots(figsize=(8, 10))

    Phylo.draw(
        tree,
        axes=ax,
        do_show=False,
        label_func=lambda clade: clade.name if clade.is_terminal() else None,
    )
    for text in ax.texts:
        text.set_fontsize(6)              # shrink labels
    ax.margins(x=0.1, y=0.05)             # extra padding around tree

    plt.tight_layout()
    plt.show()


In [ ]:
if __name__ == "__main__":
    # Read HIV RT sequences
    sequences = read_fasta("lafayette_SARS_RT.fasta")
    
    # Build distance matrix using Smith-Waterman
    dist_matrix, seq_ids = build_distance_matrix(sequences)
    
    # Generate unrooted tree using Neighbor-Joining
    tree = neighbor_joining(dist_matrix, seq_ids)
    
    # Plot tree
    plot_tree(str(tree))